# part 1.2

In [ ]:
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

# objective function
cost = np.array([
    [38000, 524000, 1204000],   # Lund
    [418000, 676000, 992000],   # Karlskrona
    [834000, 550000, 396000],   # Linkoping
    [2490000, 1964000, 1270000],# Umea
    [1066000, 498000, 616000],  # Karlstad
    [1300000, 1800000, 2400000]]) # hiring cost in each city
c = cost.flatten() # 18 variables, 15 variables are continuous, last 3 variables are integer. x1-x15 represents the proportion of 1500 hours worked from each city for
# each of the 5 projects, and x16-x18 are the number of employees hired in each city. x1-x3 are proportion of hours from malmo, gothenburg,stockholm for lund, 
# and so on.

integrality = np.zeros(18)
integrality[15:18] = 1 # last 3 variables are integer

# constraint matrix for number of hours per customer
A = np.zeros((5,18))
for i in range(5):
    A[i, i*3:i*3+3] = 1500

A_ub = np.array([500,1200,2400,3900,800])
A_lb = A_ub
customer_constraint = LinearConstraint(A, A_ub, A_lb)

# constraint matrix for number of hours worked by employees in each of the 3 offices
D = np.zeros((3, 18))

# Malmö
D[0, [0, 3, 6, 9, 12]] = 1
D[0, 15] = -1 # have to ensure that number of hours worked in each office doesnt exceed the limit of 1500 hours per employee

# Gothenburg
D[1, [1, 4, 7, 10, 13]] = 1
D[1, 16] = -1

# Stockholm
D[2, [2, 5, 8, 11, 14]] = 1
D[2, 17] = -1

D_lb = [-np.inf, -np.inf, -np.inf] 
D_ub = np.zeros(3)

employee_constraint = LinearConstraint(D, D_lb, D_ub)

# Solve MILP
result = milp(c=c, constraints=[customer_constraint, employee_constraint],
              integrality=integrality)
print(result)



        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: 16235600.0
              x: [ 3.333e-01  0.000e+00 ...  1.000e+00  2.000e+00]
 mip_node_count: 7
 mip_dual_bound: 16235600.0
        mip_gap: 0.0


In [ ]:
print(result.x)

[ 0.33333333  0.          0.          0.8        -0.          0.
  1.6        -0.         -0.          0.13333333  0.46666667  2.
 -0.          0.53333333  0.          3.          1.          2.        ]


In [ ]:
# total cost
total_cost = result.fun
print("Total cost: ", total_cost)

# number of hours worked in each office
malmo_hours = 0
for i in range(0, 15, 3):
    malmo_hours += result.x[i] * 1500
print("Malmö hours worked: ", malmo_hours)
print("Malmö employees hired: ", result.x[15])
print("slack hours from malmo:", 1500*result.x[15] - malmo_hours)

gothenburg_hours = 0
for i in range(1, 15, 3):
    gothenburg_hours += result.x[i] * 1500
print("Gothenburg hours worked: ", gothenburg_hours)
print("Gothenburg employees hired: ", result.x[16])
print("slack hours from gothenburg:", 1500*result.x[16] - gothenburg_hours)

stockholm_hours = 0
for i in range(2, 15, 3):
    stockholm_hours += result.x[i] * 1500
print("Stockholm hours worked: ", stockholm_hours)
print("Stockholm employees hired: ", result.x[17])
print("slack hours from stockholm:", 1500*result.x[17] - stockholm_hours)

# only malmo has slack hours

Total cost:  16235600.0
Malmö hours worked:  4300.0
Malmö employees hired:  3.0
slack hours from malmo: 200.0
Gothenburg hours worked:  1500.0000000000002
Gothenburg employees hired:  1.0
slack hours from gothenburg: -2.2737367544323206e-13
Stockholm hours worked:  3000.0
Stockholm employees hired:  2.0
slack hours from stockholm: 0.0


# part 1.3


## shadow price for number of hours for each customer

In [ ]:
# Shadow pricing.

# Redo the part above but increase each constraint of number of hours required by each customer by 1 unit, then compare costs.

customer = ["Lund", "Karlskrona", "Linkoping", "Umea", "Karlstad"]
# objective function
cost = np.array([
    [38000, 524000, 1204000],   # Lund
    [418000, 676000, 992000],   # Karlskrona
    [834000, 550000, 396000],   # Linkoping
    [2490000, 1964000, 1270000],# Umea
    [1066000, 498000, 616000],  # Karlstad
    [1300000, 1800000, 2400000]]) # hiring cost in each city
c = cost.flatten() # 18 variables, 15 variables are continuous, last 3 variables are integer. x1-x15 represents the proportion of 1500 hours worked from each city for
# each of the 5 projects, and x16-x18 are the number of employees hired in each city. x1-x3 are proportion of hours from malmo, gothenburg,stockholm for lund, 
# and so on.

integrality = np.zeros(18)
integrality[15:18] = 1 # last 3 variables are integer

# constraint matrix for number of hours per customer
A = np.zeros((5,18))
for i in range(5):
    A[i, i*3:i*3+3] = 1500

# plus 1 to each constraint to get new costs and get shadow pricing
for i in range(5):
    A_ub = np.array([500,1200,2400,3900,800])
    A_ub[i] +=1
    A_lb = A_ub
    customer_constraint = LinearConstraint(A, A_ub, A_lb)

    # constraint matrix for number of hours worked by employees in each of the 3 offices
    D = np.zeros((3, 18))

    # Malmö
    D[0, [0, 3, 6, 9, 12]] = 1
    D[0, 15] = -1 # have to ensure that number of hours worked in each office doesnt exceed the limit of 1500 hours per employee

    # Gothenburg
    D[1, [1, 4, 7, 10, 13]] = 1
    D[1, 16] = -1

    # Stockholm
    D[2, [2, 5, 8, 11, 14]] = 1
    D[2, 17] = -1

    D_lb = [-np.inf, -np.inf, -np.inf] 
    D_ub = np.zeros(3)

    employee_constraint = LinearConstraint(D, D_lb, D_ub)

    # Solve MILP
    result = milp(c=c, constraints=[customer_constraint, employee_constraint],
                integrality=integrality)
    print("new cost:", result.fun)
    print("shadow price for",customer[i], ":", (result.fun - total_cost))
    print("")


new cost: 16235625.333333334
shadow price for Lund : 25.333333333954215

new cost: 16235878.666666668
shadow price for Karlskrona : 278.66666666790843

new cost: 16236155.999999998
shadow price for Linkoping : 555.9999999981374

new cost: 16237260.0
shadow price for Umea : 1660.0

new cost: 16236282.666666666
shadow price for Karlstad : 682.6666666660458



## Shadow price for increasing working hours in malmo, gothenburg, stockholm

In [ ]:
# Shadow pricing.

# Redo the part above but increase each constraint of number of hours required by each customer by 1 unit, then compare costs.

office = ["Malmö", "Gothenburg", "Stockholm"]
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

# objective function
cost = np.array([
    [38000, 524000, 1204000],   # Lund
    [418000, 676000, 992000],   # Karlskrona
    [834000, 550000, 396000],   # Linkoping
    [2490000, 1964000, 1270000],# Umea
    [1066000, 498000, 616000],  # Karlstad
    [1300000, 1800000, 2400000]]) # hiring cost in each city
c = cost.flatten() # 18 variables, 15 variables are continuous, last 3 variables are integer. x1-x15 represents the proportion of 1500 hours worked from each city for
# each of the 5 projects, and x16-x18 are the number of employees hired in each city. x1-x3 are proportion of hours from malmo, gothenburg,stockholm for lund, 
# and so on.

integrality = np.zeros(18)
integrality[15:18] = 1 # last 3 variables are integer

# constraint matrix for number of hours per customer
A = np.zeros((5,18))
for i in range(5):
    A[i, i*3:i*3+3] = 1500

A_ub = np.array([500,1200,2400,3900,800])
A_lb = A_ub
customer_constraint = LinearConstraint(A, A_ub, A_lb)

for i in range(3):
    hours_per_office = [1500,1500,1500]
    hours_per_office[i] += 1 # Increase working hours by each employee in each office by 1 hour.
    # constraint matrix for number of hours worked by employees in each of the 3 offices
    D = np.zeros((3, 18))

    # Malmö
    D[0, [0, 3, 6, 9, 12]] = 1
    D[0, 15] = -hours_per_office[0]/1500 # have to ensure that number of hours worked in each office doesnt exceed the limit of 1500 hours per employee

    # Gothenburg
    D[1, [1, 4, 7, 10, 13]] = 1
    D[1, 16] = -hours_per_office[1]/1500

    # Stockholm
    D[2, [2, 5, 8, 11, 14]] = 1
    D[2, 17] = -hours_per_office[2]/1500

    D_lb = [-np.inf, -np.inf, -np.inf] 
    D_ub = np.zeros(3)

    employee_constraint = LinearConstraint(D, D_lb, D_ub)

    # Solve MILP
    result = milp(c=c, constraints=[customer_constraint, employee_constraint],
                integrality=integrality)
    print("new cost:", result.fun)
    print("shadow price for",office[i], ":", (result.fun - total_cost))
    print("") 

new cost: 16235599.999999994
shadow price for Malmö : -5.587935447692871e-09

new cost: 16235249.333333332
shadow price for Gothenburg : -350.66666666790843

new cost: 16233973.33333333
shadow price for Stockholm : -1626.666666669771



#### Only max working hours in malmo has 0 shadow price because it was the only constraint with slack. So increasing working hours wouldnt change the cost

## customer in umea requires more hours

### calculating new cost via shadow price 

In [ ]:
# Calculated umea shadow price earlier
shadow_price = 1660
earlier_required_hours = 3900
earlier_total_cost = 16235600.0

new_hours = [4100,4300,4500]
for i in range(len(new_hours)):
    new_cost = (new_hours[i] - earlier_required_hours) * shadow_price + earlier_total_cost
    print("New cost for", new_hours[i], ":", new_cost)


New cost for 4100 : 16567600.0
New cost for 4300 : 16899600.0
New cost for 4500 : 17231600.0


### Calculating new cost via solver

In [ ]:
earlier_total_cost = 16235600.0
new_hours = [4100,4300,4500]
# objective function
cost = np.array([
    [38000, 524000, 1204000],   # Lund
    [418000, 676000, 992000],   # Karlskrona
    [834000, 550000, 396000],   # Linkoping
    [2490000, 1964000, 1270000],# Umea
    [1066000, 498000, 616000],  # Karlstad
    [1300000, 1800000, 2400000]]) # hiring cost in each city
c = cost.flatten() # 18 variables, 15 variables are continuous, last 3 variables are integer. x1-x15 represents the proportion of 1500 hours worked from each city for
# each of the 5 projects, and x16-x18 are the number of employees hired in each city. x1-x3 are proportion of hours from malmo, gothenburg,stockholm for lund, 
# and so on.

integrality = np.zeros(18)
integrality[15:18] = 1 # last 3 variables are integer

# constraint matrix for number of hours per customer
A = np.zeros((5,18))
for i in range(5):
    A[i, i*3:i*3+3] = 1500
for i in range(len(new_hours)):
    A_ub = np.array([500,1200,2400,new_hours[i],800])
    A_lb = A_ub
    customer_constraint = LinearConstraint(A, A_ub, A_lb)

    # constraint matrix for number of hours worked by employees in each of the 3 offices
    D = np.zeros((3, 18))

    # Malmö
    D[0, [0, 3, 6, 9, 12]] = 1
    D[0, 15] = -1 # have to ensure that number of hours worked in each office doesnt exceed the limit of 1500 hours per employee

    # Gothenburg
    D[1, [1, 4, 7, 10, 13]] = 1
    D[1, 16] = -1

    # Stockholm
    D[2, [2, 5, 8, 11, 14]] = 1
    D[2, 17] = -1

    D_lb = [-np.inf, -np.inf, -np.inf] 
    D_ub = np.zeros(3)

    employee_constraint = LinearConstraint(D, D_lb, D_ub)

    # Solve MILP
    result = milp(c=c, constraints=[customer_constraint, employee_constraint],
                integrality=integrality)
    print("Total cost: ", result.fun)
    print("Number of employees hired in each office: ", result.x[15:18])


Total cost:  16567600.0
Number of employees hired in each office:  [3. 1. 2.]
Total cost:  18199600.0
Number of employees hired in each office:  [4. 1. 2.]
Total cost:  18460000.0
Number of employees hired in each office:  [ 4. -0.  3.]


#### The result for the 1st case is the same for the shadow price and solver method because the nature of the solution doesnt change, it is still 3 employees in malmo, 1 in gothenburg, 2 in stockholm. But the nature of the solution changes for the next 2 cases, so the shadow price cost estimate is inaccurate.

# part 1.4

In [ ]:
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

earlier_total_cost = 16235600.0

# objective function
cost = np.array([
    [38000, 524000, 1204000],   # Lund
    [418000, 676000, 992000],   # Karlskrona
    [834000, 550000, 396000],   # Linkoping
    [2490000, 1964000, 1270000],# Umea
    [1066000, 498000, 616000],  # Karlstad
    [1300000, 1800000, 2400000]]) # hiring cost in each city
c = cost.flatten() # 18 variables, 15 variables are continuous, last 3 variables are integer. x1-x15 represents the proportion of 1500 hours worked from each city for
# each of the 5 projects, and x16-x18 are the number of employees hired in each city. x1-x3 are proportion of hours from malmo, gothenburg,stockholm for lund, 
# and so on.

integrality = np.zeros(18)
integrality[15:18] = 1 # last 3 variables are integer

# constraint matrix for number of hours per customer
A = np.zeros((5,18))
for i in range(5):
    A[i, i*3:i*3+3] = 1500

A_ub = np.array([500,1200,2400,3900,800])
A_lb = A_ub
customer_constraint = LinearConstraint(A, A_ub, A_lb)

# constraint matrix for number of hours worked by employees in each of the 3 offices
D = np.zeros((3, 18))

# Malmö
D[0, [0, 3, 6, 9, 12]] = 1
D[0, 15] = -1 # have to ensure that number of hours worked in each office doesnt exceed the limit of 1500 hours per employee

# Gothenburg
D[1, [1, 4, 7, 10, 13]] = 1
D[1, 16] = -1

# Stockholm
D[2, [2, 5, 8, 11, 14]] = 1
D[2, 17] = -1

D_lb = [-np.inf, -np.inf, -np.inf] 
D_ub = np.zeros(3)

employee_constraint = LinearConstraint(D, D_lb, D_ub)

# Set bound for gothenburg employee to >= 2
lb = np.zeros(18)
ub = np.full(18, np.inf)
lb[16] = 2 
bounds = Bounds(lb, ub)

# Solve MILP
result = milp(c=c, constraints=[customer_constraint, employee_constraint],
              integrality=integrality, bounds = bounds)
print("new cost:", result.fun)
print("Difference in cost: ", result.fun - earlier_total_cost)
print("Number of employees hired in each office: ", result.x[15:18])

new cost: 16329600.0
Difference in cost:  94000.0
Number of employees hired in each office:  [3. 2. 1.]


## Cost estimate increases by 94,000 when forced to hire at least 2 employees in Gothenburg.